<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 2: Usernetes: Kubernetes in User Space</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">Bringing up a control plane inside a Flux job</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">SC26 &middot; Chicago &middot; November 2026</p>
</div>

Converged computing, on-premises. You have a cluster with Flux as the system scheduler,
and you want Kubernetes-native workloads on it without asking an administrator for a
cluster.

Usernetes is Kubernetes running entirely in user space, launched as a Flux job. Nothing
here needs a cloud account, and nothing here needs root.

<div>
<center><img src="img/flux-usernetes-turkducken.png" width="620"/></center>
</div>

Everything in Module 2 builds on this notebook. The Flux Operator in the next notebook
gets installed *into* the cluster you are about to start, and the Kubeflow Trainer after
that goes into the same one.


## 1. Start the control plane

Normally, we start Usernetes under a Flux batch job, meaning you (the user) do not have to do it, and do not see it. Here, we are going to show you all the setup. Run the following script <strong>in the terminal to your left</strong> to watch your control plane come up!

```bash
bash /home/ubuntu/start-usernetes.sh
```

When the script finishes, you'd normally export the `KUBECONFIG` to your environment. This is your credentials to interact with the cluster. However, you don't need to because we write the default to `~/.kube/config`.

```bash
export KUBECONFIG=/home/ubuntu/usernetes/kubeconfig
```

Try looking at the nodes:

```bash
kubectl get nodes
```

Right now you just have a single control plane, and we have modified it to allow for running work. This is how we map Usernetes nodes to physical nodes in HPC, with a 1:1 ratio. Finally, you can enable auto-complete (with TAB) for `kubectl`:

```bash
source <((kubectl completion bash))
```

Take a look at all the service pods! In Kubernetes we have the concept of [namespaces](https://kubernetes.io/docs/concepts/overview/working-with-objects/namespaces/) to organize things. By default we interact with `default`.

```bash
kubectl get pods --all-namespaces 
```


### Understanding the Setup

You are running on a single AWS EC2 instance with 64 cores. It is a Graviton 3 (ARM) instance. Directly installed on that instance is Flux, and the Jupyter notebook was started with `flux start --test-size=4`, meaning we are already inside of a Flux instance to emulate a batch job. We can see our Flux resources:

```bash
flux resource list
     STATE NNODES NCORES NGPUS NODELIST
      free      1     64     0 ip-10-0-25-10
 allocated      0      0     0 
      down      0      0     0 
```

We have deployed user-space Kubernetes inside of the Flux instance (job) and we can see that one Usernetes "node" is mapped to our single instance node.

```bash
kubectl get nodes -o wide
```
```console
NAME                STATUS   ROLES           AGE     VERSION   INTERNAL-IP   EXTERNAL-IP   OS-IMAGE                         KERNEL-VERSION   CONTAINER-RUNTIME
u7s-ip-10-0-25-10   Ready    control-plane   5h16m   v1.33.1   <none>        10.0.25.10    Debian GNU/Linux 12 (bookworm)   6.8.0-1029-aws   containerd://2.1.1
```

If we had a job with multiple nodes, we would see a `control-plane` role along with worker roles! For our tutorial today we just have one control plane that will also function as a worker.


### Debugging Tips 🪲

When something isn't working, here is a series of logical things to come back to look at. First, the pod logs, or metadata in YAML.

```bash
# Pod logs, one shot
kubectl logs <pod>

# Pod logs, dangling
kubectl logs <pod> -f

# YAML manifest
kubectl get pod -o yaml
```

You might also want to use `describe` to see events.

```bash
kubectl describe pods <pod>
```

Often there is an associated service not working. Look for it, and then again use logs and describe to see what might be going on.

```bash
kubectl get pods --all-namespaces
```

When you use a `Deployment` or `Job` or `Service` it is often helpful to do the equivalent `get` or `describe` for that abstraction.

<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 2, Notebook 1 complete</strong></div>

Next: [the Flux Operator, installed into this cluster](02_flux_operator.ipynb).
